# nb_data_quality — declarative rules, quarantine, severity gates
**Platform pattern:** quality rules are *data*, not code. Rules live in a `dq_rules` Delta table
(in production: Fabric SQL Database alongside `etl_entity`); this notebook is a generic runner that
never changes when a rule is added. Results land in `dq_results` (versioned, joinable to run logs);
`error`-severity failures raise and stop the pipeline, `warn`-severity failures quarantine the
offending rows and continue.

In [1]:
NOTEBOOK_NAME = "nb_data_quality"
TABLES_ROOT   = "/tmp/fabric_kit_warehouse"
RULES_TABLE   = f"{TABLES_ROOT}/_ops/dq_rules"
RESULTS_TABLE = f"{TABLES_ROOT}/_ops/dq_results"

In [2]:
# --- Session: Fabric is the default target -------------------------------------
# In Fabric you do NOT create a Spark session. The Livy layer starts it before your first
# cell runs, and `spark` (plus `sc`, `notebookutils`) are already bound. Calling
# SparkSession.builder there is at best a no-op via getOrCreate() and at worst misleading:
# master(), Delta wiring and executor shape are all decided by the Environment/pool, not here.
#
# Session-start settings belong in a %%configure -f cell ABOVE this one, or in the
# Environment's Spark properties. Only runtime-mutable keys can be set from code.
try:
    spark                      # noqa: F821  <- Fabric (and any live session): already provided
    IN_FABRIC = True
except NameError:
    # Local/dev fallback ONLY. Never runs in Fabric.
    IN_FABRIC = False
    from pyspark.sql import SparkSession
    from delta import configure_spark_with_delta_pip
    _b = (SparkSession.builder.appName(NOTEBOOK_NAME).master("local[4]")
          .config("spark.driver.memory", "2g")
          .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
          .config("spark.sql.catalog.spark_catalog",
                  "org.apache.spark.sql.delta.catalog.DeltaCatalog"))
    spark = configure_spark_with_delta_pip(_b).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print(("Fabric session (provided)" if IN_FABRIC else "local session (dev fallback)"),
      "| Spark", spark.version)

# --- session bootstrap (identical in every kit notebook; Fabric supplies `spark`) ---
import os, sys, json, time
from datetime import datetime, timezone

def get_session():
    try:
        return spark  # noqa: F821  (Fabric / existing session)
    except NameError:
        from pyspark.sql import SparkSession
        from delta import configure_spark_with_delta_pip
        b = (SparkSession.builder.appName(NOTEBOOK_NAME).master("local[4]")
             .config("spark.driver.memory", "2g")
             .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
             .config("spark.sql.catalog.spark_catalog",
                     "org.apache.spark.sql.delta.catalog.DeltaCatalog"))
        return configure_spark_with_delta_pip(b).getOrCreate()

spark = get_session()
spark.sparkContext.setLogLevel("ERROR")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
APP_ID = spark.sparkContext.applicationId
print(f"{NOTEBOOK_NAME} | run {RUN_ID} | app {APP_ID} | Spark {spark.version}")

26/08/04 12:06:04 WARN Utils: Your hostname, vm resolves to a loopback address: 127.0.0.1; using 192.0.2.2 instead (on interface eth0)
26/08/04 12:06:04 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/usr/local/lib/python3.12/dist-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-ab9f999d-b8b8-4ad5-8668-23ac817027eb;1.0
	confs: [default]


	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 420ms :: artifacts dl 13ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |   0   ||   3   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spark#spark-submit-parent-ab9f999d-b8b8-4ad5-8668-23ac817027eb
	confs: [default]
	0 artifacts copied, 3 already retrieved (0kB/13ms)


26/08/04 12:06:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


local session (dev fallback) | Spark 3.5.1
nb_data_quality | run 20260804T120613Z | app local-1785845171215 | Spark 3.5.1


In [3]:
# --- self-contained demo data (skipped if the root already has tables) ---
from pyspark.sql import functions as F
import os
os.makedirs(TABLES_ROOT, exist_ok=True)
def _has_delta(root):
    return any(os.path.isdir(os.path.join(root, d, "_delta_log")) for d in os.listdir(root)) if os.path.isdir(root) else False
if not _has_delta(TABLES_ROOT):
    (spark.range(0, 100_000)
        .withColumn("customer_id", (F.col("id") % 500).cast("int"))
        .withColumn("amount", F.round(F.rand() * 500, 2))
        .withColumn("status", F.when(F.col("id") % 7 == 0, "cancelled").otherwise("complete"))
        .withColumn("order_date", F.date_add(F.lit("2026-06-01"), (F.col("id") % 60).cast("int")))
        .repartition(24)  # deliberately many small files so the audit has something to find
        .write.format("delta").mode("overwrite").save(f"{TABLES_ROOT}/orders"))
    spark.sql(f"""CREATE TABLE IF NOT EXISTS delta.`{TABLES_ROOT}/orders_silver`
        (order_id BIGINT, customer_id INT, amount DOUBLE, status STRING, order_date DATE)
        USING DELTA TBLPROPERTIES('delta.enableDeletionVectors'='true','delta.enableChangeDataFeed'='true')""")
    src = spark.read.format("delta").load(f"{TABLES_ROOT}/orders").withColumnRenamed("id","order_id")
    src.write.format("delta").mode("append").save(f"{TABLES_ROOT}/orders_silver")
    spark.sql(f"DELETE FROM delta.`{TABLES_ROOT}/orders_silver` WHERE status='cancelled'")
    print("demo tables created: orders (24 small files), orders_silver (DV+CDF, has deletes)")
else:
    print("existing tables found - demo data skipped")

existing tables found - demo data skipped


In [4]:
# Seed demo rules once (production: INSERT into the SQL DB, never edit the notebook).
import os
from pyspark.sql import functions as F
if not os.path.isdir(os.path.join(RULES_TABLE, "_delta_log")):
    rules = [
        # rule_id, table_path, kind, column/params, severity
        (1, f"{TABLES_ROOT}/orders",        "min_rows",        json.dumps({"min": 1000}),                       "error"),
        (2, f"{TABLES_ROOT}/orders",        "not_null",        json.dumps({"column": "customer_id"}),            "error"),
        (3, f"{TABLES_ROOT}/orders_silver", "unique",          json.dumps({"column": "order_id"}),               "error"),
        (4, f"{TABLES_ROOT}/orders",        "accepted_values", json.dumps({"column": "status",
                                                                            "values": ["complete"]}),            "warn"),
        (5, f"{TABLES_ROOT}/orders_silver", "custom_sql",      json.dumps({"sql": "amount >= 0"}),               "error"),
    ]
    spark.createDataFrame(rules, ["rule_id","table_path","kind","params","severity"]) \
         .write.format("delta").mode("overwrite").save(RULES_TABLE)
    print("demo rules seeded (rule 4 is a deliberate WARN: cancelled rows exist)")

In [5]:
def run_rule(r):
    p = json.loads(r["params"])
    df = spark.read.format("delta").load(r["table_path"])
    if r["kind"] == "min_rows":
        n = df.count(); ok = n >= p["min"]; obs = f"rows={n}"
        bad = None
    elif r["kind"] == "not_null":
        bad = df.where(F.col(p["column"]).isNull())
        n = bad.count(); ok = n == 0; obs = f"nulls={n}"
    elif r["kind"] == "unique":
        bad = (df.groupBy(p["column"]).count().where("count > 1"))
        n = bad.count(); ok = n == 0; obs = f"dup_keys={n}"
    elif r["kind"] == "accepted_values":
        bad = df.where(~F.col(p["column"]).isin(p["values"]))
        n = bad.count(); ok = n == 0; obs = f"unexpected={n}"
    elif r["kind"] == "custom_sql":
        bad = df.where(f"NOT ({p['sql']})")
        n = bad.count(); ok = n == 0; obs = f"violations={n}"
    else:
        raise ValueError(f"unknown rule kind {r['kind']}")
    return ok, obs, bad

results, hard_fail = [], False
for r in spark.read.format("delta").load(RULES_TABLE).collect():
    ok, obs, bad = run_rule(r)
    status = "PASS" if ok else ("FAIL" if r["severity"] == "error" else "WARN")
    results.append({"run_id": RUN_ID, "rule_id": r["rule_id"], "table_path": r["table_path"],
                    "kind": r["kind"], "severity": r["severity"], "status": status,
                    "observed": obs, "app_id": APP_ID})
    print(f"rule {r['rule_id']:>2} {r['kind']:<16} -> {status:<4} ({obs})")
    if status == "WARN" and bad is not None and bad.columns == spark.read.format("delta").load(r["table_path"]).columns:
        qpath = r["table_path"] + "_quarantine"
        bad.withColumn("_dq_rule_id", F.lit(r["rule_id"])).withColumn("_dq_run_id", F.lit(RUN_ID)) \
           .write.format("delta").mode("append").save(qpath)
        print(f"        quarantined {bad.count()} row(s) -> {qpath}")
    hard_fail = hard_fail or status == "FAIL"

spark.createDataFrame(results).write.format("delta").mode("append").option("mergeSchema","true").save(RESULTS_TABLE)
assert not hard_fail, "error-severity DQ rule failed - stopping pipeline"
print(f"DQ complete: {sum(r['status']=='PASS' for r in results)} pass, "
      f"{sum(r['status']=='WARN' for r in results)} warn, 0 hard failures")
spark.stop() if "local" in spark.sparkContext.master else None

rule  3 unique           -> PASS (dup_keys=0)


rule  4 accepted_values  -> WARN (unexpected=14286)


        quarantined 14286 row(s) -> /tmp/fabric_kit_warehouse/orders_quarantine


rule  5 custom_sql       -> PASS (violations=0)


rule  2 not_null         -> PASS (nulls=0)


rule  1 min_rows         -> PASS (rows=100000)


DQ complete: 4 pass, 1 warn, 0 hard failures
